<a href="https://colab.research.google.com/github/pedrosouzag/edicao-imagens-ia/blob/main/semana4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!rm -rf edicao-imagens-ia
!git clone https://github.com/pedrosouzag/edicao-imagens-ia.git
%cd edicao-imagens-ia

In [ ]:
# Verificar a GPU
!nvidia-smi

In [ ]:
# Instalar dependências
!pip install -r requirements.txt

In [ ]:
import torch
from segment_anything import sam_model_registry

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")

sam.to(device)

print("SAM carregado!")
!nvidia-smi

In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionInpaintPipeline.from_pretrained("runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16)

pipe.to(device)

print("Stable Diffusion Inpainting carregado!")
!nvidia-smi

In [ ]:
from PIL import Image
import torch

def run_pipeline_with_fallback(image, prompt, mask=None):
    resolutions = [512, 256]

    for res in resolutions:
        try:
            torch.cuda.empty_cache()
            img_resized = image.resize((res, res))

            #chamada ao pipeline de inpainting
            result = sd_pipeline(prompt=prompt, image=img_resized, mask_image=mask.resize((res, res)) if mask else None, height=res, width=res).images[0]

            print(f"Sucesso em {res}x{res}")
            return result, res

        except torch.cuda.OutOfMemoryError:
            print(f"Erro em {res}x{res}, tentando resolução menor...")
            torch.cuda.empty_cache()
            continue

    raise RuntimeError("Falhou em 256x256")

In [ ]:
import os
import re
from datetime import datetime

def save_result(image, prompt, resolution):
    os.makedirs("outputs", exist_ok=True)

    # Remove caracteres especiais e troca espaços por "_"
    prompt = re.sub(r"[^\w\s]", "", prompt)
    prompt = "_".join(prompt.lower().split())[:40]

    filename = f"{datetime.now():%Y%m%d_%H%M%S}_{prompt}_{resolution}px.png"
    filepath = os.path.join("outputs", filename)

    image.save(filepath)
    print(f"Imagem salva em: {filepath}")

    return filepath